In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore")
BASE_ROOT = (
    "/content/drive/My Drive/Deep Learning Model/"
)

PREDICTION_FILE = BASE_ROOT+ "/predictions_1.csv"
DATA_Original_DIR = BASE_ROOT + "data_feat.csv"
OUTPUT_DIR = BASE_ROOT + "analysis"

DATE_COL = "dt"
ID_COL = "corp_tkr"
CLOSE_COL = "close"

HORIZON = 1
PRED_COL = f"pred_{HORIZON}"
ACTUAL_COL = f"ret_fwd_{HORIZON}"

FUTURE_WINDOWS = [1, 5, 10, 20]

MIN_STOCKS_PER_DAY = 30
MIN_OBS_FOR_MEAN = 30


In [ ]:
# ====== FactorBuilder ======

class FactorBuilder:
    def __init__(self, horizon: int):
        self.horizon = horizon
        self.ret_col = f"ret_fwd_{horizon}"

    def add_post_window_returns(self, df):
        """
        Construct post-event holding-period returns relative to the
        price at the prediction horizon.

        For each stock i at time t:
        - price_h     = Close price at t + horizon
        - price_hk    = Close price at t + horizon + k
        - ret_post_k  = (price_hk / price_h) - 1

        This design isolates returns AFTER the prediction window,
        which is useful for evaluating delayed or persistent effects.

        Parameters
        ----------
        df : pd.DataFrame
            Input dataframe containing price data.

        Returns
        -------
        pd.DataFrame
            DataFrame with post-window return columns added.
        """
        df = df.copy().sort_values([ID_COL, DATE_COL])
        g = df.groupby(ID_COL, group_keys=False)
        price_h = g[CLOSE_COL].shift(-self.horizon)
        for k in FUTURE_WINDOWS:
            price_hk = g[CLOSE_COL].shift(-(self.horizon + k))
            df[f"ret_post_{k}"] = price_hk / price_h - 1
        return df

    def add_volume_ts_cs_deviation_factors(self, df, ts_threshold=0.7, cs_threshold=0.7):
        """
        Construct volume-based deviation filters using both
        time-series (TS) and cross-sectional (CS) criteria.

        TS filter:
        - Keep deviation signal only when current volume is above
          the rolling 20-day volume quantile for the same stock.

        CS filter:
        - Keep deviation signal only when current volume ranks
          above a given percentile within the daily cross-section.

        Parameters
        ----------
        df : pd.DataFrame
            Input dataframe containing volume and deviation.
        ts_threshold : float
            Time-series quantile threshold (e.g., 0.7).
        cs_threshold : float
            Cross-sectional percentile threshold (e.g., 0.7).

        Returns
        -------
        pd.DataFrame
            DataFrame with volume-filtered deviation signals.
        """
        df = df.copy().sort_values([ID_COL, DATE_COL])
        vol_ts_q = (
            df.groupby(ID_COL)["volume"]
              .rolling(window=20, min_periods=10)
              .quantile(ts_threshold)
              .reset_index(level=0, drop=True)
        )
        df["dev_vol_ts"] = np.where(df["volume"] >= vol_ts_q, df["deviation"], np.nan)

        vol_cs_pct = df.groupby(DATE_COL)["volume"].rank(pct=True)
        df["dev_vol_cs"] = np.where(vol_cs_pct >= cs_threshold, df["deviation"], np.nan)
        return df

    def add_return_ts_cs_percentile_factors(self, df, ts_threshold=0.7, cs_threshold=0.7):
       """
        Construct return-based enhancement factors using
        time-series and cross-sectional percentile filters.

        TS filter:
        - Select observations where forward returns exceed
          the rolling historical quantile for the same stock.

        CS filter:
        - Select observations where forward returns rank
          in the top percentile of the daily cross-section.

        Parameters
        ----------
        df : pd.DataFrame
            Input dataframe containing forward returns.
        ts_threshold : float
            Time-series quantile threshold.
        cs_threshold : float
            Cross-sectional percentile threshold.

        Returns
        -------
        pd.DataFrame
            DataFrame with return-based filtered factors.
        """
        df = df.copy().sort_values([ID_COL, DATE_COL])
        ret_ts_q = (
            df.groupby(ID_COL)[self.ret_col]
              .rolling(window=20, min_periods=10)
              .quantile(ts_threshold)
              .reset_index(level=0, drop=True)
        )
        df[f"{self.ret_col}_ts_pct_filter"] = np.where(
            df[self.ret_col] >= ret_ts_q, df[self.ret_col], np.nan
        )

        ret_cs_pct = df.groupby(DATE_COL)[self.ret_col].rank(pct=True)
        df[f"{self.ret_col}_cs_pct_filter"] = np.where(
            ret_cs_pct >= cs_threshold, df[self.ret_col], np.nan
        )
        return df



In [ ]:
# ====== FactorAnalyzer ======
"""
FactorAnalyzer evaluates a factor signal using:
1) Cross-sectional quintile sorting (daily re-sorting)
2) Overlapping holding-period portfolio construction (k-day overlap)
3) Standard long-short performance metrics (Q1 - Q5)

Typical usage:
    analyzer = FactorAnalyzer(output_dir=Path("outputs"))
    qret = analyzer.compute_quintile_overlapping_returns(df, signal_col="deviation")
    analyzer.plot_quintile_cumulative_returns(qret, factor_name="deviation")
    analyzer.compute_strategy_performance(qret, factor_name="deviation")
"""
class FactorAnalyzer:
  """
    A utility class for factor evaluation via quintile portfolios.

    This class supports:
    - Assigning daily quintiles based on a signal column
    - Computing overlapping-holding portfolio returns (reduces turnover noise)
    - Plotting cumulative returns by quintile
    - Computing long-short performance statistics (Q1 - Q5)
    """

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.plot_dir = self.output_dir

    def compute_quintile_overlapping_returns(
        self,
        df,
        signal_col,
        k_holding=5,
        target_ret_col="ret_post_1",
        min_names_per_day=30,
    ):
        df = df.sort_values([ID_COL, DATE_COL]).copy()
        if signal_col not in df.columns:
            raise KeyError(f"signal_col not found: {signal_col}")
        if target_ret_col not in df.columns:
            raise KeyError(f"target_ret_col not found: {target_ret_col}")

        def assign_quintiles(group):
            g = group.copy()
            x = g[signal_col]
            valid_idx = x.dropna().index
            if len(valid_idx) < min_names_per_day:
                g["quintile"] = np.nan
                return g

            x_valid = x.loc[valid_idx]
            nunique = x_valid.nunique(dropna=True)
            if nunique < 5:
                g["quintile"] = np.nan
                return g

            try:
                g.loc[valid_idx, "quintile"] = pd.qcut(
                    x_valid,
                    q=5,
                    labels=[1, 2, 3, 4, 5],
                    duplicates="drop",
                )
            except ValueError:
                g["quintile"] = np.nan

            return g
        df = (
            df.groupby(DATE_COL, group_keys=False)
              .apply(assign_quintiles)
        )

        all_dates = sorted(df[DATE_COL].unique())
        df_by_date = {d: g for d, g in df.groupby(DATE_COL, sort=False)}

        results = []
        for i, today in enumerate(all_dates):
            daily = {f"Q{q}": 0.0 for q in range(1, 6)}
            for lb in range(k_holding):
                j = i - lb
                if j < 0:
                    continue

                g = df_by_date[all_dates[j]]

                for q in range(1, 6):
                    qret = g.loc[g["quintile"] == q, target_ret_col].mean()
                    if pd.notna(qret):
                        daily[f"Q{q}"] += qret / k_holding

            daily["date"] = today
            results.append(daily)

        return pd.DataFrame(results).set_index("date")

    def plot_quintile_cumulative_returns(self, return_df, factor_name):
        cum = (1 + return_df).cumprod()
        plt.figure(figsize=(12, 6))
        for col in cum.columns:
            plt.plot(cum.index, cum[col], label=col)
        plt.title(f"Cumulative Returns – {factor_name}")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    def compute_strategy_performance(self, return_df, factor_name, trading_days=252):
        """
        Compute long-short (Q1 - Q5) strategy performance metrics.

        Metrics
        -------
        - Annualized return (mean * trading_days)
        - Annualized Sharpe (mean/std * sqrt(trading_days))
        - t-statistic of daily returns (mean / (std/sqrt(n)))
        - Win rate (P(daily return > 0))
        - Max drawdown (from cumulative equity curve)

        Parameters
        ----------
        return_df : pd.DataFrame
            Daily quintile returns with columns including "Q1" and "Q5".
        factor_name : str
            Name of the factor/strategy (used in printed output).
        trading_days : int, default 252
            Number of trading days per year used for annualization.

        Returns
        -------
        None
            Prints performance summary to stdout.
        """
        ls = (return_df["Q1"] - return_df["Q5"]).dropna()
        n = len(ls)

        if n == 0:
            print(f"\n📌 {factor_name}")
            print("No valid LS returns.")
            return

        mean = ls.mean()
        std = ls.std(ddof=1) if n > 1 else 0.0

    # === Annual Return (simple annualization) ===
        ann_ret = mean * trading_days

    # === Annual Sharpe ===
        ann_sharpe = (mean / std) * np.sqrt(trading_days) if std > 0 else np.nan

    # === T-stat (daily mean / SE) ===
        t_stat = mean / (std / np.sqrt(n)) if std > 0 else np.nan

    # === Win Rate ===
        win_rate = (ls > 0).mean()

    # === Max Drawdown (on cumulative equity curve) ===
        equity = (1.0 + ls).cumprod()
        peak = equity.cummax()
        drawdown = equity / peak - 1.0
        max_drawdown = drawdown.min()

        print(f"\n📌 {factor_name}")
        print(f"Ann.Return:    {ann_ret:.2%}")
        print(f"Ann.Sharpe:    {ann_sharpe:.3f}")
        print(f"Max.Drawdown:  {max_drawdown:.2%}")
        print(f"T-Stat:        {t_stat:.3f}")
        print(f"Win.Rate:      {win_rate:.2%}")
        print(f"N_Days:        {n}")



In [ ]:
pred = pd.read_csv(PREDICTION_FILE, parse_dates=[DATE_COL])

In [ ]:
# ===================== Main Pipeline =====================
# This script prepares prediction outputs and raw market data,
# merges auxiliary information (volume, event count),
# and constructs the deviation factor used in downstream analysis.
pred = pd.read_csv(PREDICTION_FILE, parse_dates=[DATE_COL])
df_original=pd.read_csv(DATA_Original_DIR , parse_dates=[DATE_COL])
pred[DATE_COL] = pd.to_datetime(pred[DATE_COL])
df_original[DATE_COL] = pd.to_datetime(df_original[DATE_COL])

pred[ID_COL] = pred[ID_COL].astype(str)
df_original[ID_COL] = df_original[ID_COL].astype(str)
df_original = (
    df_original
    .sort_values([ID_COL, DATE_COL])
    .drop_duplicates(subset=[ID_COL, DATE_COL], keep="last")
)

# --------------------- Merge close-related information ---------------------
# Merge prediction results with auxiliary market information
# (volume and event_count) from the original dataset.
# many_to_one validation ensures each (ID, DATE) in pred
# maps to at most one row in df_original.
df = pred.merge(
    df_original[[DATE_COL, ID_COL, "volume",'event_count']],
    on=[DATE_COL, ID_COL],
    how="left",
    validate="many_to_one"
)

if "split_id" in df.columns:
    df = df[df["split_id"] > 0].copy()

# deviation factor
df["deviation"] = df[ACTUAL_COL] - df[PRED_COL]


In [ ]:
# ===================== Factor Construction =====================
# Build post-event returns and multiple deviation-based factor variants
# using time-series, cross-sectional, and return-based enhancement rules.
builder = FactorBuilder(HORIZON)
df = builder.add_post_window_returns(df)
df = builder.add_volume_ts_cs_deviation_factors(df)
df = builder.add_return_ts_cs_percentile_factors(df)


# ===================== Factor Analysis =====================
# Evaluate each constructed factor using quintile portfolios
# and an overlapping holding-period framework.

analyzer = FactorAnalyzer(OUTPUT_DIR)

factor_cols = [
    "deviation_evt_cs70",
    "deviation",
    "dev_vol_ts",
    "dev_vol_cs",
    f"{ACTUAL_COL}_ts_pct_filter",
    f"{ACTUAL_COL}_cs_pct_filter",
]

for col in factor_cols:
    if col not in df.columns:
        continue

    print(f"\n🔍 Analyzing factor: {col}")
    ret_df = analyzer.compute_quintile_overlapping_returns(df, col)
    analyzer.plot_quintile_cumulative_returns(ret_df, col)
    analyzer.compute_strategy_performance(ret_df, col)